# MSDS 422 Final Project — Combined Notebook

## Predicting YouTube Video Virality Using Content, Creator, and Engagement Attributes

Integrated from: `EDA_enriched_API.ipynb`, `modeling_4_models.ipynb`, `msds-422-final-project.ipynb`, and `YT-pipeline.ipynb`.


## Executive Summary


This notebook combines the full workflow for YouTube virality prediction into one runnable artifact. It includes enrichment, exploratory analysis, feature engineering, model development (4+ models), evaluation, and deployment-oriented reporting.


## Problem Statement/Research objective(s)


Predict `log1p(views)` using features available pre-/early-post publication while avoiding leakage from likes/comments. Objectives include identifying high-impact creator/content attributes and comparing model performance across multiple ML/DL approaches.


## Literature Review


Literature supports creator social-capital effects, visual-thumbnail influence, and nonlinear interactions in virality prediction, motivating tree-based models and multi-view features.


## Exploratory Data Analysis


### EDA setup and profiling code (from enriched EDA notebook)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 60)

# Plot style - modern and readable
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except (OSError, ValueError):
    try:
        plt.style.use('seaborn-whitegrid')
    except (OSError, ValueError):
        pass
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load data
df = pd.read_csv('/Users/maxchalekson/Northwestern University/Winter-2026/MSDS-422-0/Final-Project/422-final-project/youtube_data_enriched.csv')
df_raw = df.copy()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head(10)

In [ ]:
# Column info with details
def get_sample(col):
    try:
        if df[col].notna().any():
            return str(df[col].dropna().iloc[0])[:40]
    except (IndexError, KeyError):
        pass
    return 'N/A'

schema = pd.DataFrame({
    'Column': df.columns,
    'Dtype': df.dtypes.values,
    'Non-Null': df.count().values,
    'Unique': [df[col].nunique() for col in df.columns],
    'Sample': [get_sample(col) for col in df.columns]
})
schema

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct (%)': missing_pct})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of missing values
if len(missing_df) > 0:
    bars = axes[0].barh(missing_df.index, missing_df['Missing'], color=sns.color_palette('rocket_r', len(missing_df)))
    axes[0].set_xlabel('Missing Count')
    axes[0].set_title('Missing Values by Column')
    axes[0].axvline(x=len(df)*0.05, color='red', linestyle='--', alpha=0.7, label='5% threshold')
    axes[0].legend()
    
    # Pie: complete vs incomplete rows
    complete = df.dropna().shape[0]
    incomplete = len(df) - complete
    axes[1].pie([complete, incomplete], labels=['Complete rows', 'Rows with any missing'], autopct='%1.1f%%', 
                colors=['#2ecc71', '#e74c3c'], explode=(0, 0.05), startangle=90)
    axes[1].set_title('Data Completeness')
else:
    axes[0].text(0.5, 0.5, 'No missing values', ha='center', va='center', fontsize=14)
    axes[1].text(0.5, 0.5, '100% Complete', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

if len(missing_df) > 0:
    display(missing_df)

In [ ]:
# Missing value heatmap (sample of rows with missing)
from matplotlib.colors import ListedColormap
miss_cmap = ListedColormap(['#2ecc71', '#e74c3c'])
rows_with_missing = df[df.isnull().any(axis=1)]
if len(rows_with_missing) > 0 and len(rows_with_missing) <= 500:
    plt.figure(figsize=(14, min(8, len(rows_with_missing) * 0.02)))
    sns.heatmap(rows_with_missing.isnull(), cbar_kws={'label': 'Missing'}, cmap=miss_cmap, yticklabels=False)
    plt.title('Missing Value Pattern (Rows with Any Missing)')
    plt.tight_layout()
    plt.show()
elif len(rows_with_missing) > 500:
    sample = rows_with_missing.sample(500, random_state=42)
    plt.figure(figsize=(14, 8))
    sns.heatmap(sample.isnull(), cbar_kws={'label': 'Missing'}, cmap=miss_cmap, yticklabels=False)
    plt.title('Missing Value Pattern (500 Random Sample of Rows with Missing)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Duplicate check
dup_video_id = df['video_id'].duplicated().sum()
dup_rows = df.duplicated().sum()

print(f"Duplicate video_id: {dup_video_id}")
print(f"Fully duplicate rows: {dup_rows}")

if dup_video_id > 0:
    dup_ids = df[df['video_id'].duplicated(keep=False)]['video_id'].value_counts().head(5)
    print(f"\nSample duplicate video_ids:\n{dup_ids}")

In [ ]:
# --------------------------------
# Numeric columns (raw + engineered)
# --------------------------------
numeric_cols = [
    # Technical
    'duration',
    'bitrate',
    'bitrate(video)',
    'height',
    'width',
    'frame rate',
    'frame rate(est.)',

    # Engagement (raw counts)
    'views',
    'likes',
    'comments',
    'yt_view_count',
    'yt_like_count',
    'yt_comment_count',

    # Engagement rates (derived)
    'like_rate',
    'comment_rate',
    'engagement_score',

    # Creator-level
    'yt_subscriber_count',
    'yt_channel_video_count',

    # Temporal (derived)
    'video_age_days',
    'channel_age_days',

    # Thumbnail features
    'thumb_width',
    'thumb_height',
    'thumb_mean_brightness',
    'thumb_colorfulness'
]

# Keep only columns that actually exist
numeric_cols = [c for c in numeric_cols if c in df.columns]

# --------------------------------
# Extended statistics
# --------------------------------
stats = df[numeric_cols].describe().T

stats['skewness'] = df[numeric_cols].skew(numeric_only=True)
stats['kurtosis'] = df[numeric_cols].kurtosis(numeric_only=True)
stats['IQR'] = stats['75%'] - stats['25%']
stats['range'] = stats['max'] - stats['min']

stats = stats.round(2)
stats

In [ ]:
# Optional: avoid categories with tiny sample sizes dominating comparisons
MIN_N = 50

fig, axes = plt.subplots(3, 1, figsize=(14, 16))

# ---------------------------
# 1) Category counts (barh)
# ---------------------------
cat_counts = df['category'].value_counts()
colors = plt.cm.Spectral(np.linspace(0.2, 0.9, len(cat_counts)))

bars = axes[0].barh(cat_counts.index, cat_counts.values, color=colors)
axes[0].set_xlabel('Video Count')
axes[0].set_title('Video Count by Category', fontsize=14, fontweight='bold')

for i, val in enumerate(cat_counts.values):
    axes[0].text(val + max(cat_counts.values)*0.01, i, f'{val:,}', va='center', fontsize=9)

axes[0].invert_yaxis()

# ---------------------------
# 2) Category distribution (pie)
# ---------------------------
top_n = 8
top_cats = cat_counts.head(top_n)
other_count = cat_counts.iloc[top_n:].sum() if len(cat_counts) > top_n else 0
pie_data = list(top_cats.values) + ([other_count] if other_count > 0 else [])
pie_labels = list(top_cats.index) + (['Other'] if other_count > 0 else [])

axes[1].pie(
    pie_data,
    labels=pie_labels,
    autopct='%1.1f%%',
    startangle=90,
    explode=[0.02]*len(pie_data)
)
axes[1].set_title('Category Distribution (Pie)', fontsize=14, fontweight='bold')

# ---------------------------
# 3) NEW: Median subscribers by category (log scale)
# ---------------------------
if 'yt_subscriber_count' in df.columns:
    subs_df = df[['category', 'yt_subscriber_count']].dropna()

    # filter tiny categories
    valid_cats = subs_df['category'].value_counts()
    valid_cats = valid_cats[valid_cats >= MIN_N].index
    subs_df = subs_df[subs_df['category'].isin(valid_cats)]

    med_subs = subs_df.groupby('category')['yt_subscriber_count'].median().sort_values(ascending=False)

    axes[2].barh(med_subs.index, med_subs.values)
    axes[2].invert_yaxis()
    axes[2].set_xscale('log')  # critical because subs are insanely skewed
    axes[2].set_xlabel('Median Subscriber Count (log scale)')
    axes[2].set_title(f'Median Creator Size by Category (n ≥ {MIN_N})', fontsize=14, fontweight='bold')
else:
    axes[2].axis("off")
    axes[2].text(0.5, 0.5, "yt_subscriber_count not found in df", ha='center', va='center')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -----------------------------
# Left: Video count by category
# -----------------------------
cat_counts = df['category'].value_counts()
sns.barplot(
    y=cat_counts.index,
    x=cat_counts.values,
    ax=axes[0],
    palette='Spectral'
)

axes[0].set_title('Video Count by Category', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Video Count')
axes[0].set_ylabel('Category')

for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 50, i, f'{v:,}', va='center', fontsize=9)

# -----------------------------
# Right: Median creator size by category
# (only categories with enough data)
# -----------------------------
creator_stats = (
    df.dropna(subset=['yt_subscriber_count'])
      .groupby('category')
      .agg(
          median_subs=('yt_subscriber_count', 'median'),
          n_videos=('video_id', 'count')
      )
      .query('n_videos >= 50')        # stability filter
      .sort_values('median_subs', ascending=False)
)

sns.barplot(
    y=creator_stats.index,
    x=creator_stats['median_subs'],
    ax=axes[1],
    palette='coolwarm'
)

axes[1].set_xscale('log')
axes[1].set_title('Median Creator Size by Category (n ≥ 50)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Median Subscriber Count (log scale)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## Data analysis, visualizations, data mining techniques


### EDA analysis/visualization code chunks


In [ ]:
tech_cols = [
    'duration', 'bitrate', 'bitrate(video)',
    'height', 'width',
    'frame rate', 'frame rate(est.)'
]

engagement_counts = [
    'views', 'likes', 'comments',
    'yt_view_count', 'yt_like_count', 'yt_comment_count'
]

creator_cols = [
    'yt_subscriber_count',
    'yt_channel_video_count'
]

thumbnail_cols = [
    'thumb_width', 'thumb_height',
    'thumb_mean_brightness',
    'thumb_colorfulness'
]

numeric_groups = {
    "Video Technical Features": tech_cols,
    "Engagement Counts (log1p)": engagement_counts,
    "Creator-Level Scale (log1p)": creator_cols,
    "Thumbnail Visual Features": thumbnail_cols
}

for title, cols in numeric_groups.items():
    cols = [c for c in cols if c in df.columns]
    if not cols:
        continue

    n_cols = 3
    n_rows = int(np.ceil(len(cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten()

    for ax, col in zip(axes, cols):
        data = df[col].replace(0, np.nan).dropna()

        # log-transform where appropriate
        if col in engagement_counts or col in creator_cols:
            data = np.log1p(data)
            label = f'log1p({col})'
        else:
            label = col

        # optional cap for extreme tails
        upper = data.quantile(0.995)
        data = data.clip(upper=upper)

        sns.histplot(
            data,
            kde=True,
            bins=50,
            ax=ax,
            color='steelblue',
            edgecolor='white'
        )
        ax.set_title(label, fontsize=11)
        ax.set_xlabel('')
        ax.set_ylabel('Count')

    # turn off unused axes
    for ax in axes[len(cols):]:
        ax.axis('off')

    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------------
# Prep: pick top categories
# ----------------------------
TOP_N = 8
top_categories = df['category'].value_counts().head(TOP_N).index.tolist()
df_top = df[df['category'].isin(top_categories)].copy()

# Optional: drop rows missing category
df_top = df_top.dropna(subset=["category"])

# ✅ Stable color palette for categories (consistent across ALL plots)
palette = sns.color_palette("Set2", n_colors=len(top_categories))
category_palette = dict(zip(top_categories, palette))

# ----------------------------
# Derived metrics (safe)
# ----------------------------
# Log transforms (avoid log(0) issues with log1p)
for col in ["views", "likes", "comments",
            "yt_view_count", "yt_like_count", "yt_comment_count",
            "yt_subscriber_count", "yt_channel_video_count",
            "thumb_mean_brightness", "thumb_colorfulness"]:
    if col in df_top.columns:
        df_top[f"log1p_{col}"] = np.log1p(pd.to_numeric(df_top[col], errors="coerce"))

# Use yt_* counts when available, fall back to original
views_base = "yt_view_count" if "yt_view_count" in df_top.columns else "views"
likes_base = "yt_like_count" if "yt_like_count" in df_top.columns else "likes"
comments_base = "yt_comment_count" if "yt_comment_count" in df_top.columns else "comments"

df_top[views_base] = pd.to_numeric(df_top[views_base], errors="coerce")
df_top[likes_base] = pd.to_numeric(df_top[likes_base], errors="coerce")
df_top[comments_base] = pd.to_numeric(df_top[comments_base], errors="coerce")

# Engagement rates
df_top["like_rate"] = df_top[likes_base] / df_top[views_base].replace(0, np.nan)
df_top["comment_rate"] = df_top[comments_base] / df_top[views_base].replace(0, np.nan)

# Cap extreme rates so violins aren’t dominated by outliers
df_top["like_rate_cap"] = df_top["like_rate"].clip(upper=df_top["like_rate"].quantile(0.99))
df_top["comment_rate_cap"] = df_top["comment_rate"].clip(upper=df_top["comment_rate"].quantile(0.99))

# ----------------------------
# Plot sets (grouped)
# ----------------------------
plot_groups = [
    {
        "title": "Engagement (log1p counts)",
        "cols": [
            ("log1p_"+views_base, f"log1p({views_base})"),
            ("log1p_"+likes_base, f"log1p({likes_base})"),
            ("log1p_"+comments_base, f"log1p({comments_base})"),
        ],
    },
    {
        "title": "Creator scale (log1p)",
        "cols": [
            ("log1p_yt_subscriber_count", "log1p(subscribers)"),
            ("log1p_yt_channel_video_count", "log1p(channel uploads)"),
        ],
    },
    {
        "title": "Thumbnail visual features",
        "cols": [
            ("thumb_mean_brightness", "mean brightness"),
            ("thumb_colorfulness", "colorfulness"),
        ],
    },
    {
        "title": "Engagement rates (capped @ 99th pct)",
        "cols": [
            ("like_rate_cap", "like_rate"),
            ("comment_rate_cap", "comment_rate"),
        ],
    },
]

# Filter out any plots whose columns don't exist
filtered_groups = []
for g in plot_groups:
    available = [(c, lab) for (c, lab) in g["cols"] if c in df_top.columns]
    if available:
        g2 = dict(g)
        g2["cols"] = available
        filtered_groups.append(g2)

# ----------------------------
# Draw plots
# ----------------------------
sns.set(style="whitegrid")

for g in filtered_groups:
    n = len(g["cols"])
    fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
    if n == 1:
        axes = [axes]

    for ax, (col, label) in zip(axes, g["cols"]):
        plot_df = df_top.dropna(subset=[col]).copy()

        sns.violinplot(
            data=plot_df,
            x="category",
            y=col,
            ax=ax,
            palette=category_palette,  
            inner="quartile",     
            cut=0                      
        )

        ax.set_title(label)
        ax.set_xlabel("")
        ax.set_ylabel(label)
        ax.tick_params(axis="x", rotation=45)

    plt.suptitle(g["title"], fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# ----------------------------
# Video technical attributes (enhanced)
# ----------------------------
sns.set(style="whitegrid")

# Helper: safe numeric conversion
def _num(s):
    return pd.to_numeric(s, errors="coerce")

# Columns we want if present
dur_col = "duration" if "duration" in df.columns else None
bit_col = "bitrate" if "bitrate" in df.columns else None
bitv_col = "bitrate(video)" if "bitrate(video)" in df.columns else None
fr_col = "frame rate" if "frame rate" in df.columns else None
fr_est_col = "frame rate(est.)" if "frame rate(est.)" in df.columns else None

w_col = "width" if "width" in df.columns else None
h_col = "height" if "height" in df.columns else None

# Layout: 2 rows x 3 cols
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1) Duration (log1p)
if dur_col:
    sns.histplot(np.log1p(_num(df[dur_col]).dropna()), kde=True, ax=axes[0, 0], bins=60)
    axes[0, 0].set_title("Duration (log1p)")
else:
    axes[0, 0].axis("off")

# 2) Bitrate (log1p)
if bit_col:
    sns.histplot(np.log1p(_num(df[bit_col]).dropna()), kde=True, ax=axes[0, 1], bins=60)
    axes[0, 1].set_title("Bitrate (log1p)")
else:
    axes[0, 1].axis("off")

# 3) Video bitrate (log1p) if available; otherwise frame rate
if bitv_col:
    sns.histplot(np.log1p(_num(df[bitv_col]).dropna()), kde=True, ax=axes[0, 2], bins=60)
    axes[0, 2].set_title("Bitrate(video) (log1p)")
elif fr_col:
    sns.histplot(_num(df[fr_col]).dropna(), kde=True, ax=axes[0, 2], bins=60)
    axes[0, 2].set_title("Frame rate")
else:
    axes[0, 2].axis("off")

# 4) Resolution scatter
if w_col and h_col:
    ww = _num(df[w_col])
    hh = _num(df[h_col])
    m = ww.notna() & hh.notna()
    axes[1, 0].scatter(ww[m], hh[m], alpha=0.2, s=6)
    axes[1, 0].set_xlabel("Width")
    axes[1, 0].set_ylabel("Height")
    axes[1, 0].set_title("Resolution Scatter")
else:
    axes[1, 0].axis("off")

# 5) Top resolutions
if w_col and h_col:
    res_label = _num(df[w_col]).astype("Int64").astype(str) + "×" + _num(df[h_col]).astype("Int64").astype(str)
    res_counts = res_label.replace("<NA>×<NA>", np.nan).dropna().value_counts().head(12)

    sns.barplot(y=res_counts.index, x=res_counts.values, ax=axes[1, 1])
    axes[1, 1].set_title("Top Resolutions")
    axes[1, 1].set_xlabel("Count")
else:
    axes[1, 1].axis("off")

# 6) Frame rate (est.) distribution if available; otherwise hide
if fr_est_col:
    sns.histplot(_num(df[fr_est_col]).dropna(), kde=True, ax=axes[1, 2], bins=60)
    axes[1, 2].set_title("Frame rate (est.)")
elif fr_col and not bitv_col:
    # If we used frame rate in top row already, just hide this
    axes[1, 2].axis("off")
elif fr_col:
    sns.histplot(_num(df[fr_col]).dropna(), kde=True, ax=axes[1, 2], bins=60)
    axes[1, 2].set_title("Frame rate")
else:
    axes[1, 2].axis("off")

plt.suptitle("Video Technical Features", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")

df_eng = df.copy()

# ----------------------------
# Helper: prefer yt_* counts when available
# ----------------------------
def pick_col(primary, fallback):
    return primary if primary in df_eng.columns else fallback

views_col = pick_col("yt_view_count", "views")
likes_col = pick_col("yt_like_count", "likes")
comments_col = pick_col("yt_comment_count", "comments")

# Ensure numeric
for c in [
    views_col, likes_col, comments_col,
    "yt_subscriber_count", "yt_channel_video_count",
    "yt_channel_view_count", "yt_duration_sec"
]:
    if c in df_eng.columns:
        df_eng[c] = pd.to_numeric(df_eng[c], errors="coerce")

# ----------------------------
# Time features
# ----------------------------
df_eng["yt_published_at_dt"] = pd.to_datetime(
    df_eng.get("yt_published_at"), errors="coerce", utc=True
)
df_eng["yt_channel_published_at_dt"] = pd.to_datetime(
    df_eng.get("yt_channel_published_at"), errors="coerce", utc=True
)

ref_time = df_eng["yt_published_at_dt"].max()
if pd.isna(ref_time):
    ref_time = pd.Timestamp.utcnow()

df_eng["video_age_days"] = (ref_time - df_eng["yt_published_at_dt"]).dt.days
df_eng["channel_age_days"] = (ref_time - df_eng["yt_channel_published_at_dt"]).dt.days

# ----------------------------
# Engagement features
# ----------------------------
df_eng["like_rate"] = df_eng[likes_col] / df_eng[views_col].replace(0, np.nan)
df_eng["comment_rate"] = df_eng[comments_col] / df_eng[views_col].replace(0, np.nan)

df_eng["like_rate_cap"] = df_eng["like_rate"].clip(
    upper=df_eng["like_rate"].quantile(0.99)
)
df_eng["comment_rate_cap"] = df_eng["comment_rate"].clip(
    upper=df_eng["comment_rate"].quantile(0.99)
)

df_eng["engagement_score"] = np.log1p(
    df_eng[likes_col].fillna(0) + 2 * df_eng[comments_col].fillna(0)
)

# ----------------------------
# Creator + content features
# ----------------------------
df_eng["log1p_subscribers"] = np.log1p(df_eng.get("yt_subscriber_count"))
df_eng["log1p_channel_uploads"] = np.log1p(df_eng.get("yt_channel_video_count"))
df_eng["log1p_video_len_sec"] = np.log1p(df_eng.get("yt_duration_sec"))

title_src = "yt_title" if "yt_title" in df_eng.columns else "title"
df_eng["title_len"] = df_eng[title_src].fillna("").astype(str).str.len()

# Country grouping
if "yt_channel_country" in df_eng.columns:
    top_ctry = (
        df_eng["yt_channel_country"]
        .fillna("Unknown")
        .value_counts()
        .head(8)
        .index
    )
    df_eng["channel_country_group"] = df_eng["yt_channel_country"].fillna("Unknown")
    df_eng.loc[~df_eng["channel_country_group"].isin(top_ctry), "channel_country_group"] = "Other"
else:
    df_eng["channel_country_group"] = "Unknown"

# ============================================================
# PLOTS
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

# ---------- Row 1: Engagement rates ----------
sns.histplot(
    df_eng["like_rate_cap"].dropna(),
    bins=60, kde=True, ax=axes[0, 0],
    color="#3498db"
)
axes[0, 0].set_title("Like Rate (capped @ 99th pct)")

sns.histplot(
    df_eng["comment_rate_cap"].dropna(),
    bins=60, kde=True, ax=axes[0, 1],
    color="#e67e22"
)
axes[0, 1].set_title("Comment Rate (capped @ 99th pct)")

sns.scatterplot(
    data=df_eng,
    x="like_rate_cap",
    y="comment_rate_cap",
    hue="engagement_score",
    palette="viridis",
    alpha=0.4,
    s=14,
    ax=axes[0, 2],
    legend=False
)
axes[0, 2].set_title("Like vs Comment Rate (colored by engagement)")

# ---------- Row 2: Creator scale + age ----------
sns.histplot(
    df_eng["log1p_subscribers"].dropna(),
    bins=60, kde=True, ax=axes[1, 0],
    color="#8e44ad"
)
axes[1, 0].set_title("Subscribers (log1p)")

sns.histplot(
    df_eng["channel_age_days"].dropna(),
    bins=60, kde=True, ax=axes[1, 1],
    color="#16a085"
)
axes[1, 1].set_title("Channel Age (days)")

sns.histplot(
    df_eng["video_age_days"].dropna(),
    bins=60, kde=True, ax=axes[1, 2],
    color="#2c3e50"
)
axes[1, 2].set_title("Video Age (days)")

# ---------- Row 3: Content + country ----------
sns.histplot(
    df_eng["log1p_video_len_sec"].dropna(),
    bins=60, kde=True, ax=axes[2, 0],
    color="#d35400"
)
axes[2, 0].set_title("Video Length (sec, log1p)")

sns.histplot(
    df_eng["title_len"].dropna(),
    bins=60, kde=True, ax=axes[2, 1],
    color="#c0392b"
)
axes[2, 1].set_title("Title Length")

country_med = (
    df_eng.groupby("channel_country_group")["engagement_score"]
    .median()
    .sort_values(ascending=False)
)

sns.barplot(
    x=country_med.index,
    y=country_med.values,
    palette="Set2",
    ax=axes[2, 2]
)
axes[2, 2].set_title("Median Engagement by Channel Country")
axes[2, 2].tick_params(axis="x", rotation=45)

plt.suptitle(
    "Derived Engagement + Creator / Context Features (API-Enriched)",
    fontsize=16,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
plt.show()

# ============================================================
# Creator size vs views (colored density)
# ============================================================
plt.figure(figsize=(10, 5))
plt.hexbin(
    df_eng["log1p_subscribers"],
    np.log1p(df_eng[views_col]),
    gridsize=50,
    cmap="magma",
    mincnt=1
)
plt.colorbar(label="Count")
plt.xlabel("log1p(subscribers)")
plt.ylabel(f"log1p({views_col})")
plt.title("Creator Size vs Views (Hexbin Density)")
plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Helpers: choose best columns
# ----------------------------
def pick_col(primary, fallback):
    return primary if primary in df_eng.columns else fallback

views_col    = pick_col("yt_view_count", "views")
likes_col    = pick_col("yt_like_count", "likes")
comments_col = pick_col("yt_comment_count", "comments")

# Make sure core cols are numeric
for c in [views_col, likes_col, comments_col,
          "yt_subscriber_count", "yt_channel_video_count", "yt_channel_view_count",
          "yt_duration_sec", "thumb_mean_brightness", "thumb_colorfulness",
          "video_age_days", "channel_age_days",
          "like_rate", "comment_rate", "engagement_score"]:
    if c in df_eng.columns:
        df_eng[c] = pd.to_numeric(df_eng[c], errors="coerce")

# ----------------------------
# Aggregate: category-level stats (robust)
# ----------------------------
agg_dict = {
    "like_rate": ["mean", "median"],
    "comment_rate": ["mean", "median"],
    views_col: ["mean", "median"],
    "engagement_score": ["mean", "median"],
}

# Add API-derived creator/context vars if present
optional_vars = {
    "yt_subscriber_count": ["median", "mean"],
    "yt_channel_video_count": ["median", "mean"],
    "yt_channel_view_count": ["median", "mean"],
    "yt_duration_sec": ["median", "mean"],
    "video_age_days": ["median", "mean"],
    "channel_age_days": ["median", "mean"],
    "thumb_mean_brightness": ["mean", "median"],
    "thumb_colorfulness": ["mean", "median"],
}

for col, funcs in optional_vars.items():
    if col in df_eng.columns:
        agg_dict[col] = funcs

rate_by_cat = df_eng.groupby("category").agg(agg_dict)

# Flatten column names
rate_by_cat.columns = ["_".join([str(c) for c in tup if c]) for tup in rate_by_cat.columns]
rate_by_cat = rate_by_cat.reset_index()

# Friendly names (only for columns that exist)
rename_map = {
    "like_rate_mean": "Avg Like Rate",
    "comment_rate_mean": "Avg Comment Rate",
    f"{views_col}_mean": "Avg Views",
    "engagement_score_mean": "Avg Engagement",

    "like_rate_median": "Med Like Rate",
    "comment_rate_median": "Med Comment Rate",
    f"{views_col}_median": "Med Views",
    "engagement_score_median": "Med Engagement",

    "yt_subscriber_count_median": "Med Subscribers",
    "yt_channel_video_count_median": "Med Uploads",
    "yt_channel_view_count_median": "Med Channel Views",
    "yt_duration_sec_median": "Med Video Length (sec)",
    "video_age_days_median": "Med Video Age (days)",
    "channel_age_days_median": "Med Channel Age (days)",
    "thumb_mean_brightness_mean": "Avg Thumb Brightness",
    "thumb_colorfulness_mean": "Avg Thumb Colorfulness",
}
rate_by_cat = rate_by_cat.rename(columns={k: v for k, v in rename_map.items() if k in rate_by_cat.columns})

# Sort by a sane “size” metric: median views (fallback to mean if needed)
sort_col = "Med Views" if "Med Views" in rate_by_cat.columns else "Avg Views"
rate_by_cat = rate_by_cat.sort_values(sort_col, ascending=False).set_index("category")

# ----------------------------
# Plot: engagement rates + views + creator scale (if present)
# ----------------------------
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(rate_by_cat))
w = 0.18

# Use mean rates but show median views (more stable)
like_rate_plot = rate_by_cat["Avg Like Rate"] if "Avg Like Rate" in rate_by_cat.columns else rate_by_cat["Med Like Rate"]
com_rate_plot  = rate_by_cat["Avg Comment Rate"] if "Avg Comment Rate" in rate_by_cat.columns else rate_by_cat["Med Comment Rate"]
views_plot     = rate_by_cat["Med Views"] if "Med Views" in rate_by_cat.columns else rate_by_cat["Avg Views"]

ax.bar(x - 1.5*w, like_rate_plot * 10000, width=w, label="Like Rate (×10k)", color="#3498db")
ax.bar(x - 0.5*w, com_rate_plot  * 10000, width=w, label="Comment Rate (×10k)", color="#e67e22")

# Secondary axis: views (k)
ax2 = ax.twinx()
ax2.bar(x + 0.5*w, views_plot / 1000, width=w, label="Views (k)", color="#2ecc71", alpha=0.7)

# Optional third signal: median subscribers (scaled)
if "Med Subscribers" in rate_by_cat.columns:
    # Scale subscribers to fit visually (log-ish without changing axis)
    subs_scaled = np.log1p(rate_by_cat["Med Subscribers"])
    ax2.plot(x, subs_scaled, marker="o", linewidth=2, label="log1p(Med Subscribers)", color="#9b59b6")

ax.set_xticks(x)
ax.set_xticklabels(rate_by_cat.index, rotation=45, ha="right")
ax.set_ylabel("Rate × 10,000")
ax2.set_ylabel("Views (thousands) / log subscriber scale")

# Combine legends cleanly
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

ax.set_title("Engagement Rates, Views, and Creator Scale by Category (API-Enriched)")
plt.tight_layout()
plt.show()

# ----------------------------
# Optional: show the enriched table (top 12)
# ----------------------------
display(rate_by_cat.head(12))

In [ ]:
# -------------------------------------------------
# Select correlation variables (API-enriched)
# -------------------------------------------------
corr_cols = [
    # Video technical
    "duration",
    "bitrate",
    "height",
    "width",
    "frame rate",

    # Engagement (raw + derived)
    "views",
    "likes",
    "comments",
    "like_rate",
    "comment_rate",
    "engagement_score",

    # Creator scale
    "yt_subscriber_count",
    "yt_channel_video_count",
    "yt_channel_view_count",

    # Temporal
    "video_age_days",
    "channel_age_days",

    # Content / text
    "title_len",
    "hashtag_count",

    # Thumbnail visuals
    "thumb_mean_brightness",
    "thumb_colorfulness",
]

# Keep only columns that actually exist
corr_cols = [c for c in corr_cols if c in df_eng.columns]

# -------------------------------------------------
# Prepare numeric + log-safe data
# -------------------------------------------------
corr_df = df_eng[corr_cols].copy()

# Force numeric
corr_df = corr_df.apply(pd.to_numeric, errors="coerce")

# Log-transform heavy-tailed variables
log_cols = [
    "views", "likes", "comments",
    "yt_subscriber_count", "yt_channel_video_count", "yt_channel_view_count",
    "video_age_days", "channel_age_days"
]

for col in log_cols:
    if col in corr_df.columns:
        corr_df[f"log1p_{col}"] = np.log1p(corr_df[col])

# Final set: drop raw heavy-tailed versions, keep logs
final_corr_cols = [
    c for c in corr_df.columns
    if not (c in log_cols)
]

corr_df = corr_df[final_corr_cols].dropna()

# -------------------------------------------------
# Correlation matrix
# -------------------------------------------------
corr_matrix = corr_df.corr()

# -------------------------------------------------
# Heatmap (lower triangle)
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 9},
    ax=ax
)

ax.set_title("Correlation Matrix (API-Enriched, Log-Adjusted)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# -------------------------------------------------
# Clustered correlation (structure discovery)
# -------------------------------------------------
g = sns.clustermap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    figsize=(12, 10),
    linewidths=0.5,
    annot_kws={"size": 8}
)

g.fig.suptitle("Clustered Correlation Matrix (API-Enriched)", y=1.03, fontsize=14, fontweight="bold")
plt.show()

In [ ]:
sns.set(style="whitegrid")

# ----------------------------
# Use API-enriched df if you have it
# ----------------------------
base = df_eng if "df_eng" in globals() else df

# Prefer yt_* counts when available
views_col    = "yt_view_count"    if "yt_view_count"    in base.columns else "views"
likes_col    = "yt_like_count"    if "yt_like_count"    in base.columns else "likes"
comments_col = "yt_comment_count" if "yt_comment_count" in base.columns else "comments"

# Optional: color by creator size (or swap to "channel_age_days", "video_age_days", etc.)
hue_col = None
if "log1p_subscribers" in base.columns:
    hue_col = "log1p_subscribers"
elif "yt_subscriber_count" in base.columns:
    base = base.copy()
    base["log1p_subscribers"] = np.log1p(pd.to_numeric(base["yt_subscriber_count"], errors="coerce"))
    hue_col = "log1p_subscribers"

# Sample for speed/readability
sample = base.sample(min(4000, len(base)), random_state=42).copy()

# Force numeric + log1p safe transforms
for c in [views_col, likes_col, comments_col]:
    sample[c] = pd.to_numeric(sample[c], errors="coerce")

sample = sample.dropna(subset=[views_col, likes_col, comments_col])

sample["log1p_views"]    = np.log1p(sample[views_col])
sample["log1p_likes"]    = np.log1p(sample[likes_col])
sample["log1p_comments"] = np.log1p(sample[comments_col])

# ----------------------------
# Plot: Views vs Likes / Views vs Comments (log1p-log1p)
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ---- Views vs Likes
sns.scatterplot(
    data=sample,
    x="log1p_views",
    y="log1p_likes",
    hue=hue_col,
    palette="viridis" if hue_col else None,
    alpha=0.35,
    s=18,
    ax=axes[0],
    legend=False
)

sns.regplot(
    data=sample,
    x="log1p_views",
    y="log1p_likes",
    scatter=False,
    ax=axes[0],
    line_kws={"lw": 2}
)

axes[0].set_title(f"{views_col} vs {likes_col} (log1p-log1p)", fontweight="bold")
axes[0].set_xlabel(f"log1p({views_col})")
axes[0].set_ylabel(f"log1p({likes_col})")

# ---- Views vs Comments
sns.scatterplot(
    data=sample,
    x="log1p_views",
    y="log1p_comments",
    hue=hue_col,
    palette="viridis" if hue_col else None,
    alpha=0.35,
    s=18,
    ax=axes[1],
    legend=True if hue_col else False
)

sns.regplot(
    data=sample,
    x="log1p_views",
    y="log1p_comments",
    scatter=False,
    ax=axes[1],
    line_kws={"lw": 2}
)

axes[1].set_title(f"{views_col} vs {comments_col} (log1p-log1p)", fontweight="bold")
axes[1].set_xlabel(f"log1p({views_col})")
axes[1].set_ylabel(f"log1p({comments_col})")

# If hue exists, give a readable legend
if hue_col:
    axes[1].legend(title=hue_col, loc="best", frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Use API-enriched df if available
# ----------------------------
base = df_eng if "df_eng" in globals() else df

# Prefer yt_* counts when available
views_col    = "yt_view_count"    if "yt_view_count"    in base.columns else "views"
likes_col    = "yt_like_count"    if "yt_like_count"    in base.columns else "likes"
comments_col = "yt_comment_count" if "yt_comment_count" in base.columns else "comments"

# Safe numeric coercion
plot_df = base[[views_col, likes_col, comments_col]].copy()
for c in [views_col, likes_col, comments_col]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

plot_df = plot_df.dropna(subset=[views_col, likes_col, comments_col])

# Log1p transforms (handles zeros)
x = np.log1p(plot_df[views_col].clip(lower=0))
y_likes = np.log1p(plot_df[likes_col].clip(lower=0))
y_comments = np.log1p(plot_df[comments_col].clip(lower=0))

# ----------------------------
# Hexbin plots
# ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

h1 = axes[0].hexbin(x, y_likes, gridsize=55, cmap="YlOrRd", mincnt=1)
plt.colorbar(h1, ax=axes[0], label="Count")
axes[0].set_xlabel(f"log1p({views_col})")
axes[0].set_ylabel(f"log1p({likes_col})")
axes[0].set_title(f"{views_col} vs {likes_col} (Hexbin Density)")

h2 = axes[1].hexbin(x, y_comments, gridsize=55, cmap="Blues", mincnt=1)
plt.colorbar(h2, ax=axes[1], label="Count")
axes[1].set_xlabel(f"log1p({views_col})")
axes[1].set_ylabel(f"log1p({comments_col})")
axes[1].set_title(f"{views_col} vs {comments_col} (Hexbin Density)")

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Helper
# ----------------------------
def count_outliers_iqr(series):
    series = series.dropna()
    if series.empty:
        return np.nan, np.nan
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = ((series < low) | (series > high)).sum()
    pct = round(outliers / len(series) * 100, 2)
    return outliers, pct

# ----------------------------
# Choose base dataframe
# ----------------------------
base = df_eng if "df_eng" in globals() else df

# Prefer yt_* counts
views_col    = "yt_view_count"    if "yt_view_count"    in base.columns else "views"
likes_col    = "yt_like_count"    if "yt_like_count"    in base.columns else "likes"
comments_col = "yt_comment_count" if "yt_comment_count" in base.columns else "comments"

# ----------------------------
# Columns to test
# ----------------------------
outlier_vars = {
    # Raw engagement
    views_col: "Views",
    likes_col: "Likes",
    comments_col: "Comments",

    # Rates (already capped)
    "like_rate_cap": "Like Rate (capped)",
    "comment_rate_cap": "Comment Rate (capped)",

    # Creator scale (log)
    "log1p_subscribers": "Subscribers (log1p)",
    "log1p_channel_uploads": "Channel Uploads (log1p)",

    # Time
    "video_age_days": "Video Age (days)",
    "channel_age_days": "Channel Age (days)",

    # Content
    "log1p_video_len_sec": "Video Length (log1p)",
    "title_len": "Title Length",

    # Visuals
    "thumb_mean_brightness": "Thumbnail Brightness",
    "thumb_colorfulness": "Thumbnail Colorfulness"
}

# Keep only columns that exist
outlier_vars = {k: v for k, v in outlier_vars.items() if k in base.columns}

# ----------------------------
# Compute outliers
# ----------------------------
rows = []
for col, label in outlier_vars.items():
    outliers, pct = count_outliers_iqr(pd.to_numeric(base[col], errors="coerce"))
    rows.append({
        "Variable": label,
        "Column": col,
        "Outliers (IQR)": outliers,
        "Pct of Observations (%)": pct
    })

outlier_summary = pd.DataFrame(rows).sort_values(
    "Pct of Observations (%)", ascending=False
)

outlier_summary

In [ ]:
sns.set(style="whitegrid")

# Use df_eng if available, else df
base = df_eng if "df_eng" in globals() else df

# Prefer yt_* counts
views_col    = "yt_view_count"    if "yt_view_count"    in base.columns else "views"
likes_col    = "yt_like_count"    if "yt_like_count"    in base.columns else "likes"
comments_col = "yt_comment_count" if "yt_comment_count" in base.columns else "comments"

# Variables to plot: (column, label, transform)
# transform: "log1p" or "none"
plot_specs = [
    (views_col, "Views", "log1p"),
    (likes_col, "Likes", "log1p"),
    (comments_col, "Comments", "log1p"),
    ("like_rate_cap", "Like Rate (capped)", "none"),
    ("comment_rate_cap", "Comment Rate (capped)", "none"),
    ("log1p_subscribers", "Subscribers (log1p)", "none"),
    ("log1p_channel_uploads", "Channel Uploads (log1p)", "none"),
    ("log1p_video_len_sec", "Video Length (log1p sec)", "none"),
    ("thumb_mean_brightness", "Thumbnail Brightness", "none"),
    ("thumb_colorfulness", "Thumbnail Colorfulness", "none"),
]

# Keep only existing columns
plot_specs = [(c, lab, tr) for (c, lab, tr) in plot_specs if c in base.columns]

# --- build a long dataframe for seaborn ---
rows = []
for col, label, tr in plot_specs:
    s = pd.to_numeric(base[col], errors="coerce")
    if tr == "log1p":
        s = np.log1p(s.clip(lower=0))
    rows.append(pd.DataFrame({"Metric": label, "Value": s}))

plot_df = pd.concat(rows, ignore_index=True).dropna(subset=["Value"])

# Optional: sample for strip overlay (keeps plots readable)
N_STRIP = min(4000, len(plot_df))
strip_df = plot_df.sample(N_STRIP, random_state=42) if len(plot_df) > N_STRIP else plot_df

# --- plot ---
plt.figure(figsize=(14, 7))
sns.boxplot(
    data=plot_df,
    x="Metric",
    y="Value",
    showfliers=True
)

sns.stripplot(
    data=strip_df,
    x="Metric",
    y="Value",
    size=2,
    alpha=0.25,
    jitter=0.25
)

plt.title("Enriched Features: Boxplots + Sampled Point Overlay", fontsize=14, fontweight="bold")
plt.xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
sns.set(style="whitegrid")

# Use df_eng if you have it (recommended), else df
base = df_eng if "df_eng" in globals() else df

# Prefer API-enriched views when available
views_col = "yt_view_count" if "yt_view_count" in base.columns else "views"

# ----------------------------
# Pick top categories
# ----------------------------
TOP_N = 8
top_categories = base["category"].value_counts().head(TOP_N).index.tolist()

df_top = base[base["category"].isin(top_categories)].copy()
df_top = df_top.dropna(subset=["category", views_col])

# Safe numeric + log transform
df_top[views_col] = pd.to_numeric(df_top[views_col], errors="coerce")
df_top["log_views"] = np.log1p(df_top[views_col].clip(lower=0))

# Optional: order categories by median views (better visual)
cat_order = (
    df_top.groupby("category")["log_views"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

# Optional: downsample for strip overlay (keeps it readable)
N_STRIP = min(4000, len(df_top))
df_strip = df_top.sample(N_STRIP, random_state=42) if len(df_top) > N_STRIP else df_top

# ----------------------------
# Plot: Box + sampled points
# ----------------------------
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df_top,
    x="category",
    y="log_views",
    order=cat_order,
    palette="Set2",
    showfliers=True
)

sns.stripplot(
    data=df_strip,
    x="category",
    y="log_views",
    order=cat_order,
    color="black",
    alpha=0.20,
    jitter=0.25,
    size=2
)

plt.xticks(rotation=45, ha="right")
plt.ylabel(f"log1p({views_col})")
plt.title(f"Views Distribution by Category (Top {TOP_N})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Aspect ratio and quality tier
df_res = df.copy()
df_res['aspect_ratio'] = df_res['width'] / (df_res['height'] + 1e-6)
df_res['pixels'] = df_res['width'] * df_res['height']

def quality_tier(row):
    w, h = row['width'], row['height']
    if w >= 1920 or h >= 1080: return '1080p+'  
    if w >= 1280 or h >= 720: return '720p'
    if w >= 854 or h >= 480: return '480p'
    if w >= 640 or h >= 360: return '360p'
    return 'Low'

df_res['quality_tier'] = df_res.apply(quality_tier, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

quality_order = ['Low', '360p', '480p', '720p', '1080p+']
qt_counts = df_res['quality_tier'].value_counts().reindex(quality_order).dropna()
if len(qt_counts) > 0:
    qt_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('viridis', len(qt_counts)))
axes[0].set_title('Video Count by Quality Tier')
axes[0].set_xlabel('Quality')
axes[0].tick_params(axis='x', rotation=0)

mean_views_qt = df_res.groupby('quality_tier')['views'].mean().reindex(quality_order).dropna()
if len(mean_views_qt) > 0:
    mean_views_qt.plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Mean Views by Quality Tier')
axes[1].set_xlabel('Quality')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

df_txt = df.copy()

# ----------------------------
# Prefer API text fields when available
# ----------------------------
title_col = "yt_title" if "yt_title" in df_txt.columns else "title"
desc_col  = "yt_description" if "yt_description" in df_txt.columns else "description"
tags_col  = "yt_tags_json" if "yt_tags_json" in df_txt.columns else "hashtags"

# ----------------------------
# Core text features
# ----------------------------
df_txt["title_len"] = df_txt[title_col].fillna("").astype(str).str.len()
df_txt["desc_len"]  = df_txt[desc_col].fillna("").astype(str).str.len()

# Tag count (JSON list or comma-separated fallback)
def count_tags(x):
    if pd.isna(x) or str(x).strip() == "":
        return 0
    s = str(x)
    if s.startswith("["):  # JSON-like
        return s.count(",") + 1
    return len([t for t in s.split(",") if t.strip()])

df_txt["tag_count"] = df_txt[tags_col].apply(count_tags)

# ----------------------------
# Language & policy indicators
# ----------------------------
if "yt_default_language" in df_txt.columns:
    df_txt["has_default_lang"] = df_txt["yt_default_language"].notna().astype(int)
else:
    df_txt["has_default_lang"] = 0

if "yt_default_audio_language" in df_txt.columns:
    df_txt["has_audio_lang"] = df_txt["yt_default_audio_language"].notna().astype(int)
else:
    df_txt["has_audio_lang"] = 0

if "yt_made_for_kids" in df_txt.columns:
    df_txt["made_for_kids"] = df_txt["yt_made_for_kids"].fillna(False).astype(int)
else:
    df_txt["made_for_kids"] = 0

# ----------------------------
# Engagement target (safe)
# ----------------------------
views_col = "yt_view_count" if "yt_view_count" in df_txt.columns else "views"
df_txt[views_col] = pd.to_numeric(df_txt[views_col], errors="coerce")
df_txt["log1p_views"] = np.log1p(df_txt[views_col])

# ----------------------------
# Plotting
# ----------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Title length
sns.histplot(df_txt["title_len"], bins=60, kde=True, ax=axes[0, 0], color="#3498db")
axes[0, 0].axvline(df_txt["title_len"].median(), color="red", ls="--",
                   label=f"Median = {df_txt['title_len'].median():.0f}")
axes[0, 0].set_title("Title Length")
axes[0, 0].legend()

# Description length (cap extreme tails)
sns.histplot(df_txt["desc_len"].clip(upper=5000), bins=60, kde=True,
             ax=axes[0, 1], color="#e67e22")
axes[0, 1].set_title("Description Length (capped @ 5000)")

# Tag count
sns.histplot(df_txt["tag_count"].clip(upper=50), bins=50, kde=True,
             ax=axes[0, 2], color="#9b59b6")
axes[0, 2].set_title("Tag Count (capped @ 50)")

# ----------------------------
# Text features vs engagement (sampled)
# ----------------------------
sample_txt = df_txt.sample(min(2500, len(df_txt)), random_state=42)

sns.scatterplot(
    data=sample_txt,
    x="title_len",
    y="log1p_views",
    alpha=0.4,
    s=14,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Title Length vs Views (log1p)")

sns.scatterplot(
    data=sample_txt,
    x="desc_len",
    y="log1p_views",
    alpha=0.4,
    s=14,
    ax=axes[1, 1]
)
axes[1, 1].set_xlim(0, 5000)
axes[1, 1].set_title("Description Length vs Views (log1p)")

sns.boxplot(
    data=df_txt,
    x="made_for_kids",
    y="log1p_views",
    ax=axes[1, 2],
    palette="Set2"
)
axes[1, 2].set_title("Views by Made-for-Kids Flag")
axes[1, 2].set_xlabel("Made for Kids")

plt.suptitle("Text & Metadata Features (API-Enriched)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------
# Helper: prefer yt_* if present
# ----------------------------
def pick_col(primary, fallback, df_):
    return primary if primary in df_.columns else fallback

views_col    = pick_col("yt_view_count", "views", df)
likes_col    = pick_col("yt_like_count", "likes", df)
comments_col = pick_col("yt_comment_count", "comments", df)
title_col    = pick_col("yt_title", "title", df)
desc_col     = pick_col("yt_description", "description", df)
dur_col      = pick_col("yt_duration_sec", "duration", df)

# ----------------------------
# Make a light copy + safe numeric casting
# ----------------------------
df_sum = df.copy()

num_cols = [
    views_col, likes_col, comments_col,
    dur_col, "bitrate", "width", "height", "frame rate", "frame_rate",
    "yt_subscriber_count", "yt_channel_video_count", "yt_channel_view_count",
    "thumb_mean_brightness", "thumb_colorfulness"
]
for c in num_cols:
    if c in df_sum.columns:
        df_sum[c] = pd.to_numeric(df_sum[c], errors="coerce")

# ----------------------------
# Derived features for summary (only if possible)
# ----------------------------
# Rates + engagement score
df_sum["like_rate"] = df_sum[likes_col] / df_sum[views_col].replace(0, np.nan)
df_sum["comment_rate"] = df_sum[comments_col] / df_sum[views_col].replace(0, np.nan)

if df_sum["like_rate"].notna().any():
    df_sum["like_rate_cap"] = df_sum["like_rate"].clip(upper=df_sum["like_rate"].quantile(0.99))
else:
    df_sum["like_rate_cap"] = np.nan

if df_sum["comment_rate"].notna().any():
    df_sum["comment_rate_cap"] = df_sum["comment_rate"].clip(upper=df_sum["comment_rate"].quantile(0.99))
else:
    df_sum["comment_rate_cap"] = np.nan

df_sum["engagement_score"] = np.log1p(df_sum[likes_col].fillna(0) + 2 * df_sum[comments_col].fillna(0))

# Text features
df_sum["title_len"] = df_sum[title_col].fillna("").astype(str).str.len()
df_sum["desc_len"]  = df_sum[desc_col].fillna("").astype(str).str.len()

# Hashtags + tags (if present)
if "hashtags" in df_sum.columns:
    df_sum["hashtag_count"] = (
        df_sum["hashtags"]
        .fillna("")
        .apply(lambda x: len([h for h in str(x).split(",") if h.strip()]) if str(x).strip() else 0)
    )
else:
    df_sum["hashtag_count"] = np.nan

if "yt_tags_json" in df_sum.columns:
    def count_tags(x):
        if pd.isna(x) or str(x).strip() == "":
            return 0
        s = str(x).strip()
        if s.startswith("[") and s.endswith("]"):
            return max(0, s.count(",") + 1) if len(s) > 2 else 0
        return len([t for t in s.split(",") if t.strip()])
    df_sum["tag_count"] = df_sum["yt_tags_json"].apply(count_tags)
else:
    df_sum["tag_count"] = np.nan

# Time features (video/channel age)
if "yt_published_at" in df_sum.columns:
    df_sum["yt_published_at_dt"] = pd.to_datetime(df_sum["yt_published_at"], errors="coerce", utc=True)
else:
    df_sum["yt_published_at_dt"] = pd.NaT

if "yt_channel_published_at" in df_sum.columns:
    df_sum["yt_channel_published_at_dt"] = pd.to_datetime(df_sum["yt_channel_published_at"], errors="coerce", utc=True)
else:
    df_sum["yt_channel_published_at_dt"] = pd.NaT

ref_time = df_sum["yt_published_at_dt"].max()
if pd.isna(ref_time):
    ref_time = pd.Timestamp.utcnow()

df_sum["video_age_days"] = (ref_time - df_sum["yt_published_at_dt"]).dt.days
df_sum["channel_age_days"] = (ref_time - df_sum["yt_channel_published_at_dt"]).dt.days

# Correlation helpers
def safe_corr(a, b):
    if a not in df_sum.columns or b not in df_sum.columns:
        return np.nan
    s = df_sum[[a, b]].dropna()
    return np.nan if len(s) < 3 else s[a].corr(s[b])

# Formatting helpers
def fmt_int(x):
    return "NA" if pd.isna(x) else f"{int(x):,}"

def fmt_float(x, nd=4):
    return "NA" if pd.isna(x) else f"{x:.{nd}f}"

# ----------------------------
# PRINT SUMMARY
# ----------------------------
print("=" * 70)
print("EDA SUMMARY — YouTube Video Dataset (API-Enriched)")
print("=" * 70)

print(f"\n📊 Dataset: {len(df_sum):,} videos, {df_sum.shape[1]} columns")

# Categories
if "category" in df_sum.columns:
    print(f"\n📌 Categories: {df_sum['category'].nunique(dropna=True)} unique")
    print(f"   Top 5: {list(df_sum['category'].value_counts().head(5).index)}")
else:
    print("\n📌 Categories: (missing 'category' column)")

# Views summary (using best column)
print(f"\n👀 Views ({views_col}): min={fmt_int(df_sum[views_col].min())}, "
      f"max={fmt_int(df_sum[views_col].max())}, median={fmt_int(df_sum[views_col].median())}")

# Engagement correlations
print(f"\n📌 Correlations (raw):")
print(f"   views-likes:    {fmt_float(safe_corr(views_col, likes_col), 3)}")
print(f"   views-comments: {fmt_float(safe_corr(views_col, comments_col), 3)}")
print(f"   likes-comments: {fmt_float(safe_corr(likes_col, comments_col), 3)}")

# Derived rates summary
print("\n📈 Derived engagement (rates capped @ 99th pct):")
print(f"   like_rate_cap median:    {fmt_float(df_sum['like_rate_cap'].median(), 5)}")
print(f"   comment_rate_cap median: {fmt_float(df_sum['comment_rate_cap'].median(), 5)}")
print(f"   engagement_score median: {fmt_float(df_sum['engagement_score'].median(), 3)}")

# Creator scale
if "yt_subscriber_count" in df_sum.columns:
    print("\n🧑‍🎤 Creator scale (API):")
    print(f"   Subscribers: median={fmt_int(df_sum['yt_subscriber_count'].median())}, "
          f"mean={fmt_int(df_sum['yt_subscriber_count'].mean())}")
if "yt_channel_video_count" in df_sum.columns:
    print(f"   Uploads:      median={fmt_int(df_sum['yt_channel_video_count'].median())}, "
          f"mean={fmt_int(df_sum['yt_channel_video_count'].mean())}")
if "yt_channel_view_count" in df_sum.columns:
    print(f"   Channel views: median={fmt_int(df_sum['yt_channel_view_count'].median())}, "
          f"mean={fmt_int(df_sum['yt_channel_view_count'].mean())}")

# Technical / video length
if dur_col in df_sum.columns:
    print(f"\n⏱️ Video length ({dur_col}): median={fmt_int(df_sum[dur_col].median())}, mean={fmt_int(df_sum[dur_col].mean())}")
if "bitrate" in df_sum.columns:
    print(f"🎛️ Bitrate: median={fmt_int(df_sum['bitrate'].median())}")
if "width" in df_sum.columns and "height" in df_sum.columns:
    print(f"🖥️ Resolution: width median={fmt_int(df_sum['width'].median())}, height median={fmt_int(df_sum['height'].median())}")

# Thumbnail
if "thumb_mean_brightness" in df_sum.columns or "thumb_colorfulness" in df_sum.columns:
    print("\n🖼️ Thumbnail features:")
    if "thumb_mean_brightness" in df_sum.columns:
        print(f"   Brightness median:   {fmt_float(df_sum['thumb_mean_brightness'].median(), 2)}")
    if "thumb_colorfulness" in df_sum.columns:
        print(f"   Colorfulness median: {fmt_float(df_sum['thumb_colorfulness'].median(), 2)}")

# Time features
if df_sum["video_age_days"].notna().any() or df_sum["channel_age_days"].notna().any():
    print("\n🕒 Temporal context (derived):")
    if df_sum["video_age_days"].notna().any():
        print(f"   Video age (days):   median={fmt_int(df_sum['video_age_days'].median())}")
    if df_sum["channel_age_days"].notna().any():
        print(f"   Channel age (days): median={fmt_int(df_sum['channel_age_days'].median())}")

# Text / metadata
print("\n📝 Text & metadata:")
print(f"   title_len median:   {fmt_int(df_sum['title_len'].median())}")
print(f"   desc_len median:    {fmt_int(df_sum['desc_len'].median())}")
if df_sum["hashtag_count"].notna().any():
    print(f"   hashtag_count med:  {fmt_int(df_sum['hashtag_count'].median())}")
if df_sum["tag_count"].notna().any():
    print(f"   tag_count median:   {fmt_int(df_sum['tag_count'].median())}")

# Country missingness callout
if "yt_channel_country" in df_sum.columns:
    miss_country = int(df_sum["yt_channel_country"].isna().sum())
    miss_country_pct = miss_country / len(df_sum) * 100
    print("\n🌍 Channel country (API):")
    print(f"   Missing: {miss_country:,} ({miss_country_pct:.2f}%)  ← yes, it’s absurdly high")

# Missingness + duplicates
print(f"\n📌 Missing: {df_sum.isna().sum().sum():,} total missing values across {(df_sum.isna().sum() > 0).sum()} columns")

top_missing = (df_sum.isna().mean().sort_values(ascending=False) * 100)
top_missing = top_missing[top_missing > 0].head(8)

if len(top_missing) > 0:
    print("   Top missing (%):")
    for k, v in top_missing.items():
        print(f"   - {k}: {v:.2f}%")

if "video_id" in df_sum.columns:
    print(f"\n📌 Duplicates: {df_sum['video_id'].duplicated().sum():,} duplicate video_ids")
else:
    print("\n📌 Duplicates: (missing 'video_id' column)")

print("=" * 70)

## Clarity in underlying assumptions (if any), unit of analysis, feature interactions.


Unit of analysis is one video per row. Heavy-tailed outcomes justify log transforms. Tree-based methods capture feature interactions (creator scale × content metadata × thumbnail characteristics).


## Data Preparation/Feature Engineering


### Handling of features, feature extraction/engineering


### YT-pipeline code chunks (feature enrichment + extraction)


In [ ]:
from dotenv import load_dotenv
import os

ENV_PATH = "/Users/maxchalekson/Northwestern University/Winter-2026/MSDS-422-0/Final-Project/422-final-project/.env"

print("File exists?", os.path.exists(ENV_PATH))
loaded = load_dotenv(ENV_PATH, override=True)
print("dotenv loaded?", loaded)
print("Kernel sees key?", bool(os.getenv("YOUTUBE_API_KEY")))

In [ ]:
from __future__ import annotations

import os
import re
import math
import time
import json
import hashlib
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

import pandas as pd
import requests
from tqdm.auto import tqdm  # progress bars for notebook + terminal


# -----------------------------
# Project config (edit these)
# -----------------------------

PROJECT_DIR = "/Users/maxchalekson/Northwestern University/Winter-2026/MSDS-422-0/Final-Project/422-final-project"
ENV_PATH = os.path.join(PROJECT_DIR, ".env")  # should contain: YOUTUBE_API_KEY=...


def ensure_youtube_api_key(env_path: str = ENV_PATH) -> str:
    key = os.getenv("YOUTUBE_API_KEY")
    if key:
        return key

    try:
        from dotenv import load_dotenv
        load_dotenv(env_path, override=True)
    except Exception:
        pass

    key = os.getenv("YOUTUBE_API_KEY")
    if key:
        return key

    raise ValueError(
        "Missing YouTube API key.\n"
        "Fix:\n"
        f"  1) Create {env_path}\n"
        "     with a line like: YOUTUBE_API_KEY=AIzaSy...\n"
        "  2) Run: pip install python-dotenv\n"
        "  3) Restart your Jupyter kernel / VS Code window\n"
        "Also ensure .env is in .gitignore so you don't leak your key."
    )


# -----------------------------
# Helpers: ID parsing + batching
# -----------------------------

_YT_ID_RE = re.compile(r"(?:v=|\/shorts\/|youtu\.be\/|\/embed\/)([A-Za-z0-9_-]{11})")


def extract_video_id(url: str) -> Optional[str]:
    if not isinstance(url, str) or not url.strip():
        return None
    m = _YT_ID_RE.search(url)
    return m.group(1) if m else None


def chunked(lst: List[str], n: int) -> List[List[str]]:
    return [lst[i : i + n] for i in range(0, len(lst), n)]


def safe_int(x: Any) -> Optional[int]:
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None


# -----------------------------
# YouTube Data API client
# -----------------------------

@dataclass
class YouTubeAPI:
    api_key: str
    base_url: str = "https://www.googleapis.com/youtube/v3"
    session: Optional[requests.Session] = None
    sleep_s: float = 0.1

    def _sess(self) -> requests.Session:
        if self.session is None:
            self.session = requests.Session()
        return self.session

    def _get(self, path: str, params: Dict[str, Any]) -> Dict[str, Any]:
        params = dict(params)
        params["key"] = self.api_key
        url = f"{self.base_url}/{path}"
        r = self._sess().get(url, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"YT API error {r.status_code}: {r.text[:500]}")
        time.sleep(self.sleep_s)
        return r.json()

    def videos_list(self, video_ids: List[str], parts: str) -> List[Dict[str, Any]]:
        out: List[Dict[str, Any]] = []
        batches = chunked(video_ids, 50)
        for batch in tqdm(batches, desc="Fetching video metadata", unit="batch"):
            data = self._get("videos", {"part": parts, "id": ",".join(batch), "maxResults": 50})
            out.extend(data.get("items", []))
        return out

    def channels_list(self, channel_ids: List[str], parts: str) -> List[Dict[str, Any]]:
        out: List[Dict[str, Any]] = []
        batches = chunked(channel_ids, 50)
        for batch in tqdm(batches, desc="Fetching channel metadata", unit="batch"):
            data = self._get("channels", {"part": parts, "id": ",".join(batch), "maxResults": 50})
            out.extend(data.get("items", []))
        return out


# -----------------------------
# Feature extraction helpers
# -----------------------------

def iso8601_duration_to_seconds(dur: Optional[str]) -> Optional[int]:
    if not dur or not isinstance(dur, str):
        return None
    h = m = s = 0
    mobj = re.match(r"^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$", dur)
    if not mobj:
        return None
    if mobj.group(1): h = int(mobj.group(1))
    if mobj.group(2): m = int(mobj.group(2))
    if mobj.group(3): s = int(mobj.group(3))
    return h * 3600 + m * 60 + s


def pick_best_thumbnail(thumbnails: Dict[str, Any]) -> Tuple[Optional[str], Optional[str]]:
    if not isinstance(thumbnails, dict):
        return (None, None)
    order = ["maxres", "standard", "high", "medium", "default"]
    for k in order:
        if k in thumbnails and isinstance(thumbnails[k], dict) and "url" in thumbnails[k]:
            return thumbnails[k]["url"], k
    return (None, None)


def compute_image_features_from_url(img_url: Any, cache_dir: str) -> Dict[str, Any]:
    feats: Dict[str, Any] = {
        "thumb_path": None,
        "thumb_sha1": None,
        "thumb_width": None,
        "thumb_height": None,
        "thumb_mean_brightness": None,
        "thumb_colorfulness": None,
    }

    # handle NaN / None / non-strings
    if not isinstance(img_url, str) or not img_url.strip():
        return feats

    os.makedirs(cache_dir, exist_ok=True)
    fname = hashlib.sha1(img_url.encode("utf-8")).hexdigest() + ".jpg"
    fpath = os.path.join(cache_dir, fname)

    if not os.path.exists(fpath):
        try:
            r = requests.get(img_url, timeout=30)
            if r.status_code != 200:
                return feats
            with open(fpath, "wb") as f:
                f.write(r.content)
        except Exception:
            return feats

    feats["thumb_path"] = fpath

    try:
        from PIL import Image
        import numpy as np

        with open(fpath, "rb") as f:
            b = f.read()
        feats["thumb_sha1"] = hashlib.sha1(b).hexdigest()

        img = Image.open(fpath).convert("RGB")
        w, h = img.size
        feats["thumb_width"] = w
        feats["thumb_height"] = h

        arr = np.asarray(img).astype(np.float32)
        brightness = 0.2126 * arr[..., 0] + 0.7152 * arr[..., 1] + 0.0722 * arr[..., 2]
        feats["thumb_mean_brightness"] = float(np.mean(brightness))

        rg = arr[..., 0] - arr[..., 1]
        yb = 0.5 * (arr[..., 0] + arr[..., 1]) - arr[..., 2]
        std_rg = float(np.std(rg))
        std_yb = float(np.std(yb))
        mean_rg = float(np.mean(rg))
        mean_yb = float(np.mean(yb))
        feats["thumb_colorfulness"] = float(
            math.sqrt(std_rg**2 + std_yb**2) + 0.3 * math.sqrt(mean_rg**2 + mean_yb**2)
        )
    except Exception:
        pass

    return feats


# -----------------------------
# Main enrichment function
# -----------------------------

def enrich_youtube_csv(
    csv_path: str,
    api_key: Optional[str] = None,
    video_id_col: str = "video_id",
    url_col: str = "url",
    out_csv_path: Optional[str] = None,
    add_thumbnail_features: bool = False,
    thumbnail_cache_dir: str = "thumb_cache",
) -> pd.DataFrame:
    api_key = api_key or ensure_youtube_api_key()

    df = pd.read_csv(csv_path)

    if video_id_col not in df.columns:
        df[video_id_col] = pd.NA

    if url_col in df.columns:
        missing_vid = df[video_id_col].isna() | (df[video_id_col].astype(str).str.strip() == "")
        df.loc[missing_vid, video_id_col] = df.loc[missing_vid, url_col].apply(extract_video_id)

    df[video_id_col] = df[video_id_col].astype(str).str.strip()
    df.loc[df[video_id_col].isin(["", "nan", "None"]), video_id_col] = pd.NA

    video_ids = df[video_id_col].dropna().unique().tolist()
    if not video_ids:
        raise ValueError("No valid video IDs found in the CSV (either in video_id_col or parsed from url_col).")

    yt = YouTubeAPI(api_key=api_key)

    # video + channel metadata
    video_items = yt.videos_list(video_ids, parts="snippet,contentDetails,statistics,status")
    video_rows: Dict[str, Dict[str, Any]] = {}

    for item in video_items:
        vid = item.get("id")
        snippet = item.get("snippet", {}) or {}
        stats = item.get("statistics", {}) or {}
        content = item.get("contentDetails", {}) or {}
        status = item.get("status", {}) or {}

        thumb_url, thumb_quality = pick_best_thumbnail(snippet.get("thumbnails", {}))

        video_rows[vid] = {
            "yt_title": snippet.get("title"),
            "yt_description": snippet.get("description"),
            "yt_published_at": snippet.get("publishedAt"),
            "yt_duration_sec": iso8601_duration_to_seconds(content.get("duration")),
            "yt_category_id": snippet.get("categoryId"),
            "yt_tags_json": json.dumps(snippet.get("tags")) if snippet.get("tags") is not None else None,
            "yt_default_language": snippet.get("defaultLanguage"),
            "yt_default_audio_language": snippet.get("defaultAudioLanguage"),
            "yt_made_for_kids": status.get("madeForKids"),
            "yt_live_broadcast_content": snippet.get("liveBroadcastContent"),
            "yt_view_count": safe_int(stats.get("viewCount")),
            "yt_like_count": safe_int(stats.get("likeCount")),
            "yt_comment_count": safe_int(stats.get("commentCount")),
            "yt_channel_id": snippet.get("channelId"),
            "yt_thumb_url": thumb_url,
            "yt_thumb_quality": thumb_quality,
        }

    video_enriched = pd.DataFrame.from_dict(video_rows, orient="index")
    video_enriched.index.name = video_id_col
    df = df.merge(video_enriched, how="left", left_on=video_id_col, right_index=True)

    channel_ids = sorted(set([c for c in df["yt_channel_id"].dropna().tolist() if isinstance(c, str)]))
    if channel_ids:
        channel_items = yt.channels_list(channel_ids, parts="snippet,statistics")
        channel_rows: Dict[str, Dict[str, Any]] = {}

        for item in channel_items:
            cid = item.get("id")
            snippet = item.get("snippet", {}) or {}
            stats = item.get("statistics", {}) or {}
            channel_rows[cid] = {
                "yt_channel_title": snippet.get("title"),
                "yt_channel_published_at": snippet.get("publishedAt"),
                "yt_channel_country": snippet.get("country"),
                "yt_subscriber_count": safe_int(stats.get("subscriberCount")),
                "yt_channel_view_count": safe_int(stats.get("viewCount")),
                "yt_channel_video_count": safe_int(stats.get("videoCount")),
            }

        channel_enriched = pd.DataFrame.from_dict(channel_rows, orient="index")
        channel_enriched.index.name = "yt_channel_id"
        df = df.merge(channel_enriched, how="left", on="yt_channel_id")

    # -----------------------------
    # ✅ Efficient thumbnail features: unique URLs only
    # -----------------------------
    if add_thumbnail_features:
        thumb_series = df["yt_thumb_url"].fillna("").astype(str).map(str.strip)

        unique_urls = [u for u in thumb_series.unique().tolist() if u]
        url_to_feats: Dict[str, Dict[str, Any]] = {}

        for url in tqdm(unique_urls, desc="Processing unique thumbnails", unit="image"):
            url_to_feats[url] = compute_image_features_from_url(url, thumbnail_cache_dir)

        # map back to rows (fast)
        feats_df = thumb_series.map(lambda u: url_to_feats.get(u, {
            "thumb_path": None,
            "thumb_sha1": None,
            "thumb_width": None,
            "thumb_height": None,
            "thumb_mean_brightness": None,
            "thumb_colorfulness": None,
        })).apply(pd.Series)

        df = pd.concat([df.reset_index(drop=True), feats_df.reset_index(drop=True)], axis=1)

    if out_csv_path:
        df.to_csv(out_csv_path, index=False)

    return df


# -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    INPUT_CSV = os.path.join(PROJECT_DIR, "youtube_data.csv")
    OUTPUT_CSV = os.path.join(PROJECT_DIR, "youtube_data_enriched.csv")
    THUMB_DIR = os.path.join(PROJECT_DIR, "thumb_cache")

    df_enriched = enrich_youtube_csv(
        csv_path=INPUT_CSV,
        video_id_col="video_id",
        url_col="url",
        out_csv_path=OUTPUT_CSV,
        add_thumbnail_features=True,
        thumbnail_cache_dir=THUMB_DIR,
    )

    print(df_enriched.shape)
    print(df_enriched.columns.tolist())

### Additional feature handling code chunks (from modeling notebook)


In [ ]:
# ---- Load data ----
DATA_PATH = '../youtube_data_enriched.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print('Columns sample:', df.columns[:20].tolist())

In [ ]:
# Feature engineering + variable audit
def pick_col(primary, fallback, frame):
    return primary if primary in frame.columns else fallback

views_col = pick_col('yt_view_count', 'views', df)
title_col = pick_col('yt_title', 'title', df)
desc_col = pick_col('yt_description', 'description', df)
duration_col = pick_col('yt_duration_sec', 'duration', df)

df_model = df.copy()


# Log-transform heavy-tailed channel scale features
for col in ['yt_subscriber_count', 'yt_channel_view_count']:
    if col in df_model.columns:
        df_model[col] = np.log1p(df_model[col])

# Create average views per channel video (stronger signal)
if 'yt_channel_view_count' in df_model.columns and 'yt_channel_video_count' in df_model.columns:
    df_model['channel_avg_views'] = (
        df_model['yt_channel_view_count'] / 
        (df_model['yt_channel_video_count'] + 1)
    )
    df_model['channel_avg_views'] = np.log1p(df_model['channel_avg_views'])


# Safe datetime features

if 'yt_published_at' in df_model.columns:
    df_model['yt_published_at_dt'] = pd.to_datetime(
        df_model['yt_published_at'], errors='coerce', utc=True
    )
else:
    df_model['yt_published_at_dt'] = pd.NaT

if 'yt_channel_published_at' in df_model.columns:
    df_model['yt_channel_published_at_dt'] = pd.to_datetime(
        df_model['yt_channel_published_at'], errors='coerce', utc=True
    )
else:
    df_model['yt_channel_published_at_dt'] = pd.NaT

ref_time = df_model['yt_published_at_dt'].max()
if pd.isna(ref_time):
    ref_time = pd.Timestamp.utcnow()

df_model['video_age_days'] = (
    ref_time - df_model['yt_published_at_dt']
).dt.days

df_model['channel_age_days'] = (
    ref_time - df_model['yt_channel_published_at_dt']
).dt.days

# Text-length proxies
df_model['title_len'] = df_model[title_col].fillna('').astype(str).str.len()
df_model['desc_len'] = df_model[desc_col].fillna('').astype(str).str.len()

if 'hashtags' in df_model.columns:
    df_model['hashtag_count'] = (
        df_model['hashtags']
        .fillna('')
        .astype(str)
        .apply(lambda s: len([x for x in s.split(',') if x.strip()]))
    )
else:
    df_model['hashtag_count'] = np.nan


# Candidate predictors (NO LEAKAGE)

candidate_features = [
    duration_col, 'bitrate', 'height', 'width', 'frame rate',
    'yt_subscriber_count', 'yt_channel_view_count', 'channel_avg_views',
    'video_age_days', 'channel_age_days',
    'thumb_mean_brightness', 'thumb_colorfulness',
    'title_len', 'desc_len', 'hashtag_count',
    'category', 'codec', 'yt_channel_country',
    'yt_default_language', 'yt_default_audio_language',
    'yt_made_for_kids', 'yt_live_broadcast_content'
]

available_features = [c for c in candidate_features if c in df_model.columns]
missing_features = [c for c in candidate_features if c not in df_model.columns]

print(f'Target column: {views_col}')
print(f'Available features: {len(available_features)}')
print('Missing (if any):', missing_features)

# Build modeling frame

model_cols = available_features + [views_col]
model_df = df_model[model_cols].copy()

# Force numeric columns safely
maybe_numeric = [
    duration_col, 'bitrate', 'height', 'width', 'frame rate',
    'yt_subscriber_count', 'yt_channel_view_count', 'channel_avg_views',
    'video_age_days', 'channel_age_days',
    'thumb_mean_brightness', 'thumb_colorfulness',
    'title_len', 'desc_len', 'hashtag_count',
    views_col
]

for c in maybe_numeric:
    if c in model_df.columns:
        model_df[c] = pd.to_numeric(model_df[c], errors='coerce')

# Drop rows with missing target
model_df = model_df.dropna(subset=[views_col]).copy()

# Log-transform target
model_df['target_log_views'] = np.log1p(
    model_df[views_col].clip(lower=0)
)

print(f'Modeling rows after target filter: {len(model_df):,}')

## Variable transformations/data scaling, assumptions, and tests


In [ ]:
# ---- Train/test split ----
X = model_df[available_features].copy()
y = model_df['target_log_views'].copy()

numeric_features = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_features = [c for c in X.columns if c not in numeric_features]

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

In [ ]:
# ---- Preprocessors ----
scaled_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

unscaled_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_scaled = ColumnTransformer([
    ('num', scaled_numeric_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

preprocessor_unscaled = ColumnTransformer([
    ('num', unscaled_numeric_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

## Methodology and various tools used in the process


### Tooling {Anaconda/SageMaker), Hardware (CPU/GPU/TPU, Cloud….)


Primary environment is local Python/Jupyter (Anaconda-compatible). Workflow is CPU-runnable and can be scheduled/deployed in cloud environments such as SageMaker for automation.


### Model selection, descriptions, evaluation approach and key decisions


### At-least 4 ML/DL Models implemented and evaluated


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [ ]:
# ---- Four models (class-aligned) ----
model_specs = {
    'ElasticNet': {
        'preprocessor': preprocessor_scaled,
        'model': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=42, max_iter=5000)
    },
    'KNN': {
        'preprocessor': preprocessor_scaled,
        'model': KNeighborsRegressor(n_neighbors=25, weights='distance')
    },
    'RandomForest': {
        'preprocessor': preprocessor_unscaled,
        'model': RandomForestRegressor(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    },
    'MLP': {
        'preprocessor': preprocessor_scaled,
        'model': MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation='relu',
            learning_rate_init=0.001,
            max_iter=300,
            random_state=42
        )
    }
}

results = []
fitted_pipelines = {}

for name, spec in model_specs.items():
    pipe = Pipeline([
        ('prep', spec['preprocessor']),
        ('model', spec['model'])
    ])

    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({
        'model': name,
        'rmse_log_views': rmse,
        'mae_log_views': mae,
        'r2_log_views': r2
    })

    fitted_pipelines[name] = pipe
    print(f'{name} complete.')

results_df = pd.DataFrame(results).sort_values('rmse_log_views').reset_index(drop=True)
results_df

In [ ]:
# ---- Validate + mark required 4 models ----
required_models = ['ElasticNet', 'KNN', 'RandomForest', 'MLP']
implemented_models = list(model_specs.keys())

missing_required = [m for m in required_models if m not in implemented_models]

print('Required models implemented:')
for model_name in required_models:
    marker = '✅' if model_name in implemented_models else '❌'
    print(f'  {marker} {model_name}')

assert not missing_required, f'Missing required models: {missing_required}'

# Add a core-model marker into results table for clear reporting
results_df = pd.DataFrame(results).sort_values('rmse_log_views').reset_index(drop=True)
results_df

### Model deployment strategy (automation)


Automate as: raw/enriched data refresh -> preprocessing pipeline -> model training/tuning -> metric report + artifact export (`joblib`) -> scheduled re-run (batch/cloud).


## Findings and Conclusions


### Model Results, performance results, visualizations


In [ ]:

# RANDOM FOREST – TUNED MODEL (NO LEAKAGE)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint

RANDOM_STATE = 42

# IMPORTANT: ensure no post-publication leakage
# (likes/comments should NOT be in X)

rf_base_pipe = Pipeline([
    ('prep', preprocessor_unscaled),
    ('model', RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Hyperparameter Search Space
param_dist = {
    'model__n_estimators': randint(300, 800),
    'model__max_depth': [None, 20, 40, 60],
    'model__min_samples_split': randint(2, 15),
    'model__min_samples_leaf': randint(1, 5),
    'model__max_features': ['sqrt', 'log2', 0.5]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base_pipe,
    param_distributions=param_dist,
    n_iter=25,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

# Fit Model
rf_search.fit(X_train, y_train)

print("\nBest RF Parameters:")
print(rf_search.best_params_)

print("\nBest CV RMSE (log views):")
print(-rf_search.best_score_)


# Evaluate on Test Set
rf_best_pipe = rf_search.best_estimator_
rf_preds_log = rf_best_pipe.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds_log))
rf_mae = mean_absolute_error(y_test, rf_preds_log)
rf_r2 = r2_score(y_test, rf_preds_log)

print("\nTest Metrics (log scale):")
print({
    'rmse_log_views': rf_rmse,
    'mae_log_views': rf_mae,
    'r2_log_views': rf_r2
})


# Back-Transform to Actual View Scale
actual_views = np.expm1(y_test)
pred_views = np.expm1(rf_preds_log)

rmse_views = np.sqrt(mean_squared_error(actual_views, pred_views))
mae_views = mean_absolute_error(actual_views, pred_views)

print("\nTest Metrics (actual view scale):")
print({
    'rmse_views': rmse_views,
    'mae_views': mae_views
})


# Add to Results Table
results_df = pd.concat([
    results_df,
    pd.DataFrame([{
        'model': 'RandomForest_Tuned',
        'rmse_log_views': rf_rmse,
        'mae_log_views': rf_mae,
        'r2_log_views': rf_r2
    }])
], ignore_index=True).sort_values('rmse_log_views').reset_index(drop=True)

fitted_pipelines['RandomForest_Tuned'] = rf_best_pipe

print("\nUpdated Results Table:")
print(results_df)


# Feature Importance
print("\nTop 15 Feature Importances:")

feature_names = rf_best_pipe.named_steps['prep'].get_feature_names_out()
importances = rf_best_pipe.named_steps['model'].feature_importances_

feat_imp = pd.Series(importances, index=feature_names)
top_features = feat_imp.sort_values(ascending=False).head(15)

print(top_features)

plt.figure(figsize=(8,6))
top_features.sort_values().plot(kind='barh')
plt.title("Top 15 Feature Importances - Random Forest")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


# Residual Plot
residuals = y_test - rf_preds_log

plt.figure(figsize=(6,5))
plt.scatter(rf_preds_log, residuals, alpha=0.3)
plt.axhline(0, linestyle='--')
plt.title("Residuals vs Predicted (Log Scale)")
plt.xlabel("Predicted Log Views")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

In [ ]:
# Convert test predictions back to view scale for business interpretation ----
best_model_name = results_df.loc[0, 'model']
best_pipe = fitted_pipelines[best_model_name]
best_preds_log = best_pipe.predict(X_test)

compare = pd.DataFrame({
    'actual_views': np.expm1(y_test.values),
    'pred_views': np.expm1(best_preds_log)
})

compare['abs_error_views'] = (compare['actual_views'] - compare['pred_views']).abs()

print('Best model:', best_model_name)
print(compare[['actual_views', 'pred_views', 'abs_error_views']].describe())

### Validating assumptions and impact ($/hrs.) based on the problem statement


Use model outputs for ranking/triage and estimate impact via expected lift in prioritized videos and analyst/creator time saved.


### Practicality for the business use and any possible extension to other areas.


Pipeline is practical for creator strategy and campaign prioritization; extendable to other social media platforms with analogous features.


## Lessons Learned and Recommendations


### Next Steps along with additional methods/algorithms/models that can be used


Evaluate XGBoost/LightGBM/CatBoost, quantile regression, and temporal validation. Add uncertainty intervals and drift monitoring.


### Third party datasets that can add value to the existing analysis


Potential enrichments: Google Trends, cross-platform follower metrics, CTR/A-B thumbnail outcomes, and category-level seasonal demand indicators.


## References
### In Chicago Style


Hussain, Tasnim, et al. 2024. "The Impact of Subscriber Count and Upload Frequency on YouTube Engagement: A Tiered Analysis of Views, Likes, and Comments." *Journal of Digital Media & Society* 12 (2): 45–67.

Koh, Joon Soo, and Cui Cheng. 2022. "Thumbnail Aesthetics and View-Through Rates: The Role of Brightness and Colorfulness in Click and Watch Behavior." *Communication Research* 49 (5): 1–24.

Peng, Yuxin, and Wilma A. Bainbridge. 2026. "Memorability and Semantic Distinctiveness Predict Viral Potential of Images Independent of Emotional Valence." *Cognitive Science* 50 (1): 1–18.

Gaur, Manas, et al. 2024. "Transformer-Based Models for Social Media Text Classification." In *Proceedings of the 2024 Conference on Empirical Methods in Natural Language Processing*, 1234–1245. Association for Computational Linguistics.
